# Scaffolding Permit Compliance Map — NYC MVP

Cross-references VLM-based scaffolding detections against NYC DoB permit data,
using **building footprint polygons** (buffered to cover sidewalk + street) for
precise spatial matching instead of arbitrary radius circles.

**Approach:** Building → scaffolding detection matching:  
For each building with a scaffold/shed filing, check if our detections confirm
scaffolding nearby. Detections far from any permitted building are flagged as
potentially unpermitted.

| Status | Color | Meaning |
|--------|-------|---------|
| **Permitted** | Green | Detection within buffered footprint of actively-permitted building |
| **Expired** | Orange | Detection near building whose permit expired or work was signed off |
| **Unpermitted** | Red | Detection not within any scaffold-permitted building's footprint |

**Data sources:**
1. Scaffolding detections via Qwen3-VL-Reranker-8B cross-encoder (`outputs/rerank/`)
2. NYC building footprints by BIN (`data/geo/nyc_buildings.parquet`)
3. DoB NOW Build Job Application Filings (Socrata `w9ak-ipjd`)
4. DoB Permit Issuance (Socrata `ipu4-2q9a`) for expiration dates

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import json
from pathlib import Path
from datetime import datetime
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Paths ---
RERANK_DIR = Path('/share/pierson/matt/mllmsci/outputs/rerank')
BUILDINGS_PATH = Path('/share/pierson/matt/mllmsci/data/geo/nyc_buildings.parquet')
CACHE_DIR = Path('/share/pierson/matt/mllmsci/notebooks/scaffolding/cache')
OUTPUT_DIR = Path('/share/pierson/matt/mllmsci/notebooks/scaffolding')
CACHE_DIR.mkdir(exist_ok=True)

# --- DoB Socrata API endpoints ---
DOB_FILINGS_URL = 'https://data.cityofnewyork.us/resource/w9ak-ipjd.json'
DOB_PERMITS_URL = 'https://data.cityofnewyork.us/resource/ipu4-2q9a.json'

# --- Building footprint buffer (State Plane feet) ---
# 20 ft = sidewalk scaffold zone only (matches 6.5% of camera detections)
# 50 ft = sidewalk + near-side street lane (~85% match rate)
# 80 ft = full street crossing (~95% match rate)
# The camera is in the roadway, typically 20-60 ft from the building edge.
BUFFER_FT = 50

NYC_SP = 'EPSG:2263'   # NY State Plane Long Island (US feet)
WGS84 = 'EPSG:4326'
TODAY = pd.Timestamp.now(tz='UTC')

print(f'Analysis date:    {TODAY.strftime("%Y-%m-%d")}')
print(f'Building buffer:  {BUFFER_FT} ft ({BUFFER_FT / 3.28084:.1f}m)')

## 1. Load Scaffolding Detections

In [ ]:
rerank_files = sorted(RERANK_DIR.glob('*.parquet'))
dfs = []
for f in rerank_files:
    df = pd.read_parquet(f)
    df['source_run'] = f.stem
    dfs.append(df)
    print(f'  {f.name}: {len(df):,} rows, '
          f'rerank_score in [{df.rerank_score.min():.3f}, {df.rerank_score.max():.3f}]')

det_raw = pd.concat(dfs, ignore_index=True)

# Per recording location, keep highest rerank score
det = (
    det_raw
    .sort_values('rerank_score', ascending=False)
    .drop_duplicates(subset='recording_id', keep='first')
    [['sample_id', 'recording_id', 'face', 'lat', 'lon',
      'retrieval_score', 'rerank_score', 'recordedAt',
      'recorderDirection', 'yawDegrees', 'source_run',
      'image_path', 'image_path_original']]
    .dropna(subset=['lat', 'lon'])
    .reset_index(drop=True)
)

det_gdf = gpd.GeoDataFrame(det, geometry=gpd.points_from_xy(det.lon, det.lat), crs=WGS84)
det_sp = det_gdf.to_crs(NYC_SP)
print(f'\nDetection locations: {len(det_gdf):,}')

## 2. Fetch DoB Permit Data

In [ ]:
def fetch_socrata(url, where, select, limit=50000, cache_key=None):
    """Paginated Socrata SODA API fetch with local parquet cache."""
    if cache_key:
        cache_path = CACHE_DIR / f'{cache_key}.parquet'
        if cache_path.exists():
            df = pd.read_parquet(cache_path)
            print(f'  Loaded {len(df):,} rows from cache ({cache_path.name})')
            return df
    records, offset = [], 0
    while True:
        params = {'$where': where, '$select': select, '$limit': limit,
                  '$offset': offset, '$order': ':id'}
        r = requests.get(url, params=params, timeout=120)
        r.raise_for_status()
        batch = r.json()
        if not batch: break
        records.extend(batch)
        print(f'  Fetched {len(records):,}...', end='\r')
        offset += limit
        if len(batch) < limit: break
    df = pd.DataFrame(records)
    print(f'  Fetched {len(records):,} records total.    ')
    if cache_key and len(df) > 0:
        df.to_parquet(CACHE_DIR / f'{cache_key}.parquet', index=False)
    return df

In [ ]:
FILINGS_COLS = ','.join([
    'job_filing_number', 'filing_status', 'filing_date',
    'first_permit_date', 'current_status_date', 'signoff_date',
    'latitude', 'longitude', 'scaffold', 'shed',
    'borough', 'house_no', 'street_name', 'block', 'lot', 'bin',
    'initial_cost', 'job_type'
])

print('Fetching DoB scaffold/shed filings (citywide)...')
filings = fetch_socrata(
    DOB_FILINGS_URL,
    where="(scaffold='1' OR shed='1') AND latitude IS NOT NULL",
    select=FILINGS_COLS,
    cache_key='dob_scaffold_shed_filings_v1'
)

for col in ['latitude', 'longitude']:
    filings[col] = pd.to_numeric(filings[col], errors='coerce')
for col in ['filing_date', 'first_permit_date', 'current_status_date', 'signoff_date']:
    filings[col] = pd.to_datetime(filings[col], errors='coerce', utc=True)

filings = filings.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
print(f'Filings: {len(filings):,}')

## 3. Building Footprints + Permit Status

Load 1M+ NYC building footprint polygons (by BIN), join with scaffold/shed
filings, and fetch permit expiration dates from Dataset 2 via BIN.
Then buffer each building footprint to cover the sidewalk + near-street zone.

In [ ]:
# Load building footprints
buildings_all = gpd.read_parquet(BUILDINGS_PATH)
print(f'NYC building footprints: {len(buildings_all):,}')

# Per BIN, take the most recent scaffold/shed filing
filings_by_bin = (
    filings
    .sort_values('filing_date', ascending=False)
    .drop_duplicates(subset='bin', keep='first')
)
print(f'Unique BINs with scaffold/shed filings: {len(filings_by_bin):,}')

# Inner join: buildings that have scaffold/shed filings
filing_cols = [
    'bin', 'job_filing_number', 'filing_date', 'first_permit_date',
    'signoff_date', 'scaffold', 'shed', 'house_no', 'street_name',
    'borough', 'filing_status'
]
buildings = buildings_all.merge(filings_by_bin[filing_cols], on='bin', how='inner')
print(f'Buildings with scaffold/shed filings: {len(buildings):,}')
print(f'By borough: {dict(buildings.borough.value_counts())}')

In [ ]:
# Fetch permit expiration dates (Dataset 2) by BIN
matched_bins = buildings['bin'].dropna().unique()
PERMITS_COLS = ','.join([
    'job__', 'bin__', 'permit_type', 'permit_subtype', 'permit_status',
    'issuance_date', 'expiration_date'
])

cache_path = CACHE_DIR / 'dob_permits_by_bin_v1.parquet'
if cache_path.exists():
    permits_df = pd.read_parquet(cache_path)
    print(f'Loaded {len(permits_df):,} permit records from cache')
else:
    BATCH_SIZE = 40
    all_records = []
    n_batches = (len(matched_bins) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(matched_bins), BATCH_SIZE):
        batch = matched_bins[i:i + BATCH_SIZE]
        or_clauses = ' OR '.join(f"bin__='{b}'" for b in batch)
        params = {'$where': f'({or_clauses}) AND issuance_date IS NOT NULL',
                  '$select': PERMITS_COLS, '$limit': 50000, '$order': ':id'}
        r = requests.get(DOB_PERMITS_URL, params=params, timeout=120)
        r.raise_for_status()
        all_records.extend(r.json())
        print(f'  Batch {i//BATCH_SIZE+1}/{n_batches}: {len(all_records):,}', end='\r')
    permits_df = pd.DataFrame(all_records)
    if len(permits_df) > 0:
        permits_df.to_parquet(cache_path, index=False)
    print(f'\nFetched {len(permits_df):,} permits for {len(matched_bins):,} BINs')

# Per BIN, latest shed-related permit
if len(permits_df) > 0:
    for col in ['issuance_date', 'expiration_date']:
        permits_df[col] = pd.to_datetime(permits_df[col], errors='coerce', utc=True)
    shed_types = {'SH', 'SD', 'SF'}
    src = permits_df[permits_df.permit_subtype.isin(shed_types)]
    if len(src) == 0:
        src = permits_df
    permits_latest = (src.sort_values('issuance_date', ascending=False)
                      .drop_duplicates(subset='bin__', keep='first')
                      [['bin__', 'issuance_date', 'expiration_date', 'permit_status', 'permit_subtype']])
else:
    permits_latest = pd.DataFrame(columns=['bin__', 'issuance_date', 'expiration_date',
                                           'permit_status', 'permit_subtype'])
print(f'BINs with expiration data: {len(permits_latest):,}')

In [ ]:
# Merge expiration into buildings and derive lifecycle status
buildings = buildings.merge(permits_latest, left_on='bin', right_on='bin__', how='left')

def derive_status(row):
    if pd.notna(row.get('expiration_date')):
        return 'active' if row['expiration_date'] > TODAY else 'expired'
    if pd.notna(row.get('signoff_date')):
        return 'completed'
    if pd.notna(row.get('first_permit_date')):
        return 'active_no_expiry'
    return 'no_permit'

buildings['permit_lifecycle'] = buildings.apply(derive_status, axis=1)
print('Building permit lifecycle:')
print(buildings.permit_lifecycle.value_counts().to_string())

# Buffer footprints in State Plane
buildings_sp = buildings.to_crs(NYC_SP)
buildings_sp['geometry_original'] = buildings_sp.geometry.copy()
buildings_sp['geometry'] = buildings_sp.geometry.buffer(BUFFER_FT)
print(f'\nBuffered {len(buildings_sp):,} building footprints by {BUFFER_FT} ft')

## 4. Building-Footprint Spatial Join

For each detection, check if it falls within a buffered building footprint
that has a scaffold/shed filing. This replaces the arbitrary 100m radius
with shape-aware matching.

In [ ]:
# Buffer sweep — how many detections match at each buffer distance?
print(f'{"Buffer":>10s} | {"Matched":>8s} | {"Unmatched":>10s} | {"Match %":>8s}')
print('-' * 50)
for bf in [20, 40, 50, 60, 80, 100, 150]:
    bldg_buf = buildings.to_crs(NYC_SP).copy()
    bldg_buf['geometry'] = bldg_buf.geometry.buffer(bf)
    m = gpd.sjoin(det_sp, bldg_buf[['geometry']], how='left', predicate='within')
    m = m.drop_duplicates(subset='recording_id', keep='first')
    n_m = m['index_right'].notna().sum()
    n_u = len(m) - n_m
    print(f'{bf:>7d} ft | {n_m:>8,} | {n_u:>10,} | {100*n_m/len(m):>7.1f}%')
print(f'\n\u2192 Using BUFFER_FT = {BUFFER_FT} ft')

In [ ]:
# Spatial join with the configured buffer
join_cols = [
    'geometry', 'bin', 'job_filing_number', 'filing_date', 'first_permit_date',
    'signoff_date', 'expiration_date', 'permit_lifecycle',
    'scaffold', 'shed', 'house_no', 'street_name', 'borough',
    'height_roof', 'filing_status'
]
# Only keep columns that exist
join_cols = [c for c in join_cols if c in buildings_sp.columns]

matched = gpd.sjoin(
    det_sp,
    buildings_sp[join_cols],
    how='left',
    predicate='within'
)

# If a detection falls in multiple overlapping buffers, keep most recent filing
matched = (
    matched
    .sort_values('filing_date', ascending=False, na_position='last')
    .drop_duplicates(subset='recording_id', keep='first')
)

# Restore datetime types
for col in ['filing_date', 'first_permit_date', 'signoff_date', 'expiration_date']:
    if col in matched.columns:
        matched[col] = pd.to_datetime(matched[col], errors='coerce', utc=True)

has_building = matched['index_right'].notna()
print(f'Detections matched to a building: {has_building.sum():,}')
print(f'Detections unmatched:             {(~has_building).sum():,}')

In [ ]:
# Classify
def classify(row):
    if pd.isna(row.get('index_right')):
        return 'unpermitted'
    lc = row.get('permit_lifecycle')
    if lc in ('active', 'active_no_expiry'):
        return 'permitted'
    if lc in ('expired', 'completed'):
        return 'expired'
    return 'unpermitted'  # filing exists but no permit issued

matched['compliance'] = matched.apply(classify, axis=1)

print('=' * 55)
print('SCAFFOLDING PERMIT COMPLIANCE (BUILDING-FOOTPRINT JOIN)')
print('=' * 55)
for status in ['permitted', 'expired', 'unpermitted']:
    n = (matched.compliance == status).sum()
    pct = 100 * n / len(matched)
    print(f'  {status:12s}: {n:5d}  ({pct:5.1f}%)')
print(f'  {"total":12s}: {len(matched):5d}')

## 5. Building-Centric View

Invert the perspective: for each building with a scaffold/shed permit,
do our embeddings detect scaffolding nearby?

| Building Status | Detection? | Interpretation |
|-----------------|-----------|----------------|
| Active permit | Yes | Confirmed — scaffold detected + permitted |
| Active permit | No | Permitted but not detected (removed? occluded?) |
| Expired permit | Yes | Lingering — scaffold still standing |
| Expired permit | No | Resolved — scaffold removed |
| No permit issued | — | Filing without action |

In [ ]:
# For each building with a filing, check if any detection falls in its buffer
det_in_buildings = gpd.sjoin(
    buildings_sp[['geometry', 'bin', 'permit_lifecycle']],
    det_sp[['geometry']],
    how='left',
    predicate='contains'
)

# Count detections per building
det_counts = (
    det_in_buildings
    .groupby('bin')
    .agg(n_detections=('index_right', lambda x: x.notna().sum()))
    .reset_index()
)

buildings_status = buildings[['bin', 'permit_lifecycle', 'borough',
                              'house_no', 'street_name']].drop_duplicates(subset='bin')
buildings_status = buildings_status.merge(det_counts, on='bin', how='left')
buildings_status['n_detections'] = buildings_status['n_detections'].fillna(0).astype(int)
buildings_status['has_detection'] = buildings_status['n_detections'] > 0

# Cross-tabulate
xtab = pd.crosstab(
    buildings_status.permit_lifecycle,
    buildings_status.has_detection,
    margins=True
)
xtab.columns = ['No Detection', 'Detection', 'Total']
print('Building Permit Status \u00d7 Detection Presence')
print('=' * 55)
print(xtab.to_string())

# Key metrics
active_bldgs = buildings_status[buildings_status.permit_lifecycle.isin(['active', 'active_no_expiry'])]
expired_bldgs = buildings_status[buildings_status.permit_lifecycle.isin(['expired', 'completed'])]
print(f'\n\u2192 Active-permit buildings with detection: '
      f'{active_bldgs.has_detection.sum():,} / {len(active_bldgs):,} '
      f'({100*active_bldgs.has_detection.mean():.1f}%)')
print(f'\u2192 Expired-permit buildings with detection: '
      f'{expired_bldgs.has_detection.sum():,} / {len(expired_bldgs):,} '
      f'({100*expired_bldgs.has_detection.mean():.1f}%) \u2190 lingering scaffolds')

## 6. Summary Statistics

In [ ]:
COLORS = {'permitted': '#2ecc71', 'expired': '#f39c12', 'unpermitted': '#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Compliance bar chart
counts = matched.compliance.value_counts().reindex(['permitted', 'expired', 'unpermitted'])
bars = axes[0].bar(counts.index, counts.values,
                   color=[COLORS[c] for c in counts.index], edgecolor='white', width=0.6)
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, count + 8,
                f'{count}\n({100*count/len(matched):.1f}%)',
                ha='center', va='bottom', fontsize=10)
axes[0].set_ylabel('Detection Count')
axes[0].set_title(f'Compliance (buffer={BUFFER_FT}ft)')
axes[0].set_ylim(0, counts.max() * 1.25)

# (b) Rerank score by compliance
for status in ['permitted', 'expired', 'unpermitted']:
    subset = matched[matched.compliance == status]
    if len(subset):
        axes[1].hist(subset.rerank_score, bins=25, alpha=0.55,
                     label=f'{status} (n={len(subset)})', color=COLORS[status])
axes[1].set_xlabel('Rerank Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Detection Confidence by Status')
axes[1].legend(fontsize=9)

# (c) Building-centric: active vs expired detection rates
for lc, color, label in [
    (['active', 'active_no_expiry'], '#2ecc71', 'Active permit'),
    (['expired', 'completed'], '#f39c12', 'Expired permit'),
]:
    sub = buildings_status[buildings_status.permit_lifecycle.isin(lc)]
    det_rate = sub.has_detection.mean() * 100 if len(sub) > 0 else 0
    axes[2].bar(label, det_rate, color=color, edgecolor='white', width=0.5)
    axes[2].text(axes[2].patches[-1].get_x() + 0.25, det_rate + 0.5,
                f'{det_rate:.1f}%', ha='center', fontsize=11)
axes[2].set_ylabel('% Buildings with Detection')
axes[2].set_title('Building-Centric Detection Rate')
axes[2].set_ylim(0, max(20, axes[2].get_ylim()[1] * 1.2))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'compliance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Interactive Compliance Map

- **Detection points**: colored by compliance status
- **DoB filing heatmap** (toggle): permit density
- Click markers for building address, permit details, and detection info

In [ ]:
det_map = matched.to_crs(WGS84)

m = folium.Map(location=[40.78, -73.96], zoom_start=12, tiles='CartoDB positron')

# Layer: DoB filing density
heat_data = filings[['latitude', 'longitude']].dropna().values.tolist()
HeatMap(heat_data, name='DoB Filing Density', min_opacity=0.25,
        radius=12, blur=15, show=False).add_to(m)

# Layers: detections by compliance
for status in ['unpermitted', 'expired', 'permitted']:
    color = COLORS[status]
    n = (det_map.compliance == status).sum()
    group = folium.FeatureGroup(name=f'Detections \u2014 {status.title()} ({n})')
    subset = det_map[det_map.compliance == status]

    for _, row in subset.iterrows():
        popup = [f"<b style='color:{color}'>{status.upper()}</b>",
                 f"<b>Recording:</b> {row.recording_id} ({row.face})",
                 f"<b>Score:</b> {row.rerank_score:.3f}"]
        if pd.notna(row.get('index_right')):
            popup += [
                '<hr style="margin:4px 0">',
                f"<b>Building BIN:</b> {row.get('bin', 'N/A')}",
                f"<b>Address:</b> {row.get('house_no', '')} {row.get('street_name', '')}",
                f"<b>Filed:</b> {str(row.filing_date)[:10] if pd.notna(row.get('filing_date')) else 'N/A'}",
                f"<b>Permit:</b> {str(row.first_permit_date)[:10] if pd.notna(row.get('first_permit_date')) else 'N/A'}",
                f"<b>Expires:</b> {str(row.expiration_date)[:10] if pd.notna(row.get('expiration_date')) else 'N/A'}",
                f"<b>Signed off:</b> {str(row.signoff_date)[:10] if pd.notna(row.get('signoff_date')) else 'N/A'}",
                f"<b>Status:</b> {row.get('permit_lifecycle', 'N/A')}",
            ]
        folium.CircleMarker(
            location=[row.lat, row.lon], radius=7,
            color=color, fill=True, fill_color=color, fill_opacity=0.75,
            popup=folium.Popup('<br>'.join(popup), max_width=350),
            tooltip=f'{status} | score={row.rerank_score:.2f}',
        ).add_to(group)
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:2px solid #bbb; font-size:13px; line-height:1.7;
     box-shadow:0 2px 6px rgba(0,0,0,0.15);">
<b>Scaffolding Compliance</b><br>
<span style="background:#2ecc71;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Permitted<br>
<span style="background:#f39c12;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Expired / completed<br>
<span style="background:#e74c3c;width:12px;height:12px;display:inline-block;
      border-radius:50%;margin-right:4px"></span> Unpermitted<br>
<hr style="margin:4px 0">
<small>Matched via building footprint + %dft buffer</small>
</div>
''' % BUFFER_FT
m.get_root().html.add_child(folium.Element(legend_html))

map_path = OUTPUT_DIR / 'scaffolding_compliance_map.html'
m.save(str(map_path))
print(f'Map saved: {map_path}')
m

## 8. Export Results

In [ ]:
export_cols = [
    'recording_id', 'sample_id', 'face', 'lat', 'lon',
    'rerank_score', 'retrieval_score', 'source_run',
    'compliance',
    'bin', 'job_filing_number', 'filing_date', 'first_permit_date',
    'expiration_date', 'signoff_date', 'permit_lifecycle',
    'scaffold', 'shed', 'house_no', 'street_name', 'borough',
    'height_roof', 'image_path', 'image_path_original',
]
export_cols = [c for c in export_cols if c in matched.columns]
export_df = matched[export_cols].copy()

export_path = OUTPUT_DIR / 'scaffolding_compliance_classified.parquet'
export_df.to_parquet(export_path, index=False)

print(f'Exported {len(export_df):,} classified detections to {export_path}')
print(f'\nCompliance:')
print(export_df.compliance.value_counts().to_string())